In [ ]:
%%local
%%html
<style>pre { white-space: pre !important; }</style>

In [ ]:
%load_ext sagemaker_studio_analytics_extension.magics
%sm_analytics emr connect  \
    --cluster-id j-11S1ZE3VHPTC4 \
    --auth-type Basic_Access \
    --language python \
    --emr-execution-role-arn arn:aws:iam::430118852718:role/BBVARole-ada-us-east-1-sbx-live-s-size-dev

## 1. Libraries

In [ ]:
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim

from mercury.contrib.ml.nn.torch import run_with_torch_distributor

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

In [ ]:
from dataproc_sdk.dataproc_sdk_datiopysparksession.datiopysparksession import DatioPysparkSession

In [ ]:
dataproc = DatioPysparkSession().get_or_create()
spark = dataproc.getSparkSession()

# 1.1 Reading Data

In [ ]:
PATH_TRAIN="s3://ada-us-east-1-sbx-live-mx-czl2-data/data/sandboxes/czl2/data/analytical/AmesH/proyecto_ames_train.csv"
PATH_TEST="s3://ada-us-east-1-sbx-live-mx-czl2-data/data/sandboxes/czl2/data/analytical/AmesH/proyecto_ames_test_respuesta.csv"

In [ ]:
%pip install awswrangler

In [ ]:
import awswrangler as wr

In [ ]:
raw_data_train = wr.s3.read_csv(path=[PATH_TRAIN])
raw_data_test = wr.s3.read_csv(path=[PATH_TEST])

In [ ]:
raw_data_train.columns = [col.lower().replace(" ", "_") for col in raw_data_train.columns]
raw_data_test.columns = [col.lower().replace(" ", "_") for col in raw_data_test.columns]

In [ ]:
for col in raw_data_train.columns:
    if (raw_data_train[col].dtype == "object") or (raw_data_train[col].dtype == "str"):
        raw_data_train[col] = raw_data_train[col].fillna("None")

for col in raw_data_test.columns:
    if (raw_data_test[col].dtype == "object") or (raw_data_test[col].dtype == "str"):
        raw_data_test[col] = raw_data_test[col].fillna("None")

## 1.1 Processing Data

In [ ]:
cols_to_drop = [
    "alley",
    "pool_qc",
    "fence",
    "misc_feature",
    "garage_yr_blt",
    "mo_sold",
    "yr_sold",
]

In [ ]:
CURRENT_YEAR = 2026

In [ ]:
raw_data_train["house_age"] = CURRENT_YEAR - raw_data_train["year_built"]
raw_data_train["years_since_remodel"] = CURRENT_YEAR - raw_data_train["year_remod/add"]
raw_data_train["total_sg"] = (
    raw_data_train["total_bsmt_sf"]
    + raw_data_train["1st_flr_sf"]
    + raw_data_train["2nd_flr_sf"]
)
raw_data_train["total_bathrooms"] = (
    raw_data_train["full_bath"]
    + 0.5 * raw_data_train["half_bath"]
    + raw_data_train["bsmt_full_bath"]
    + 0.5 * raw_data_train["bsmt_half_bath"]
)
raw_data_train["is_pool"] = np.where(raw_data_train["pool_area"] > 0, 1, 0)
raw_data_train["is_fence"] = np.where(raw_data_train["fence"].isnull(), 0, 1)

In [ ]:
raw_data_test["house_age"] = CURRENT_YEAR - raw_data_test["year_built"]
raw_data_test["years_since_remodel"] = CURRENT_YEAR - raw_data_test["year_remod/add"]
raw_data_test["total_sg"] = (
    raw_data_test["total_bsmt_sf"]
    + raw_data_test["1st_flr_sf"]
    + raw_data_test["2nd_flr_sf"]
)
raw_data_test["total_bathrooms"] = (
    raw_data_test["full_bath"]
    + 0.5 * raw_data_test["half_bath"]
    + raw_data_test["bsmt_full_bath"]
    + 0.5 * raw_data_test["bsmt_half_bath"]
)
raw_data_test["is_pool"] = np.where(raw_data_test["pool_area"] > 0, 1, 0)
raw_data_test["is_fence"] = np.where(raw_data_test["fence"].isnull(), 0, 1)

In [ ]:
columns_to_drop = [
    "year_built",
    "year_remod/add",
    "total_bsmt_sf",
    "1st_flr_sf",
    "2nd_flr_sf",
    "full_bath",
    "half_bath",
    "bsmt_full_bath",
    "bsmt_half_bath",
    "pool_area",
]

In [ ]:
raw_data_test.info()

In [ ]:
X_train = raw_data_train.drop(columns=columns_to_drop + ["saleprice"], axis=1)
y_train = raw_data_train["saleprice"]

X_test = raw_data_test.drop(columns=columns_to_drop + ["saleprice"], axis=1)
y_test = raw_data_test["saleprice"]

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

In [ ]:
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()

In [ ]:
numeric_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
        ("one_encoder", one_hot_encoder),
    ]
)

In [ ]:
fe_pipe = Pipeline(
    [
        (
            "etl",
            ColumnTransformer(
                [
                    ("numerical", numeric_transformer, numerical_cols),
                    ("categorial", categorical_transformer, categorical_cols),
                ]
            ),
        )
    ]
)

In [ ]:
X_train_transformed = fe_pipe.fit_transform(X_train)
X_test_transformed = fe_pipe.transform(X_test)

In [ ]:
cat_encoder = (
    fe_pipe
    .named_steps["etl"]
    .named_transformers_["categorial"]
    .named_steps["one_encoder"]
)

cat_features = cat_encoder.get_feature_names_out(categorical_cols)
feature_names = np.concatenate([numerical_cols, cat_features])

In [ ]:
X_train_df = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index,
)

X_test_df = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=X_test.index,
)

In [ ]:
X_train_df.describe().T

In [ ]:
print("X_train_df:", X_train_df.shape)
print("X_test_df:", X_test_df.shape)

In [ ]:
X_train_df["saleprice"] = y_train.values
X_test_df["saleprice"] = y_test.values

X_train_np = X_train_df.drop("saleprice", axis=1).values.astype("float32")
y_train_np = X_train_df["saleprice"].values.astype("float32")

X_test_np = X_test_df.drop("saleprice", axis=1).values.astype("float32")
y_test_np = X_test_df["saleprice"].values.astype("float32")

input_dim = X_train_np.shape[1]

print("input_dim:", input_dim)

## 1.3 Model 

In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout, activation):
        super().__init__()

        activations = {
            "relu": nn.ReLU,
            "elu": nn.ELU,
            "gelu": nn.GELU,
            "leaky_relu": nn.LeakyReLU,
        }

        if activation not in activations:
            raise ValueError(f"Unsupported activation: {activation}")

        layers = []
        prev_dim = input_dim
        act_fn = activations[activation]

        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(act_fn())
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.Dropout(dropout))
            prev_dim = h

        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


## 1.4 Optuna Loggers

Note: some errors arise give the Optuna logging system 

In [ ]:
import logging
import optuna

# Evita que Optuna use handlers heredados con streams cerrados
optuna.logging.disable_default_handler()
optuna.logging.disable_propagation()

# Opcional: silenciar Optuna casi por completo
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Limpia handlers problemáticos del logger raíz
root_logger = logging.getLogger()
for handler in list(root_logger.handlers):
    try:
        stream = getattr(handler, "stream", None)
        if stream is not None and getattr(stream, "closed", False):
            root_logger.removeHandler(handler)
    except Exception:
        root_logger.removeHandler(handler)

## 1.5 Aux Functions

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(False)

In [ ]:
def get_dist_info():
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    return rank, local_rank, world_size


In [ ]:
def setup_distributed():
    rank, local_rank, world_size = get_dist_info()
    if world_size > 1 and not dist.is_initialized():
        dist.init_process_group(backend="gloo")
    return rank, local_rank, world_size


def cleanup_distributed():
    if dist.is_available() and dist.is_initialized():
        try:
            dist.barrier()
        except Exception:
            pass
        try:
            dist.destroy_process_group()
        except Exception:
            pass

def reduce_sum(value, device):
    tensor = torch.tensor(value, dtype=torch.float64, device=device)
    if dist.is_available() and dist.is_initialized():
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    return tensor.item()


In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    total_items = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()

        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size

    total_loss = reduce_sum(total_loss, device)
    total_items = reduce_sum(total_items, device)
    return total_loss / max(total_items, 1)



In [ ]:

@torch.no_grad()
def evaluate_loss(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_items = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        preds = model(xb)
        loss = loss_fn(preds, yb)

        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size

    total_loss = reduce_sum(total_loss, device)
    total_items = reduce_sum(total_items, device)
    return total_loss / max(total_items, 1)



In [ ]:

@torch.no_grad()
def predict_numpy(model, loader, device):
    model.eval()
    preds = []
    y_true = []

    for xb, yb in loader:
        xb = xb.to(device)
        preds.append(model(xb).cpu().numpy())
        y_true.append(yb.cpu().numpy())

    return np.vstack(preds).ravel(), np.vstack(y_true).ravel()



In [ ]:
def train_ames_distributed(
    X_train_np,
    y_train_np,
    X_test_np,
    y_test_np,
    input_dim,
    hidden_dims,
    dropout,
    activation,
    lr,
    batch_size,
    max_epochs,
    patience,
    seed,
):
    rank, local_rank, world_size = setup_distributed()

    try:
        seed_everything(seed + rank)
        device = torch.device("cpu")

        X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32).view(-1, 1)
        X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test_np, dtype=torch.float32).view(-1, 1)

        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

        train_sampler = DistributedSampler(
            train_dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=True,
            seed=seed,
        ) if world_size > 1 else None

        val_sampler = DistributedSampler(
            test_dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=False,
            seed=seed,
        ) if world_size > 1 else None

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=(train_sampler is None),
            sampler=train_sampler,
            num_workers=0,
        )

        val_loader = DataLoader(
            test_dataset,
            batch_size=batch_size * 4,
            shuffle=False,
            sampler=val_sampler,
            num_workers=0,
        )

        eval_loader = DataLoader(
            test_dataset,
            batch_size=batch_size * 4,
            shuffle=False,
            num_workers=0,
        )

        base_model = MLPRegressor(
            input_dim=input_dim,
            hidden_dims=list(hidden_dims),
            dropout=dropout,
            activation=activation,
        ).to(device)

        model = DDP(base_model) if world_size > 1 else base_model
        loss_fn = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        best_val_loss = float("inf")
        best_state_dict = None
        epochs_without_improvement = 0
        history = []

        for epoch in range(max_epochs):
            if train_sampler is not None:
                train_sampler.set_epoch(epoch)

            train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
            val_loss = evaluate_loss(model, val_loader, loss_fn, device)

            if rank == 0:
                history.append({
                    "epoch": epoch + 1,
                    "train_loss": float(train_loss),
                    "val_loss": float(val_loss),
                })

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_state_dict = {
                        k: v.detach().cpu().clone()
                        for k, v in base_model.state_dict().items()
                    }
                    epochs_without_improvement = 0
                else:
                    epochs_without_improvement += 1

                stop_training = epochs_without_improvement >= patience
            else:
                stop_training = False

            if dist.is_available() and dist.is_initialized():
                stop_tensor = torch.tensor(int(stop_training), device=device)
                dist.broadcast(stop_tensor, src=0)
                stop_training = bool(stop_tensor.item())

            if stop_training:
                break

        #if dist.is_available() and dist.is_initialized():
        #    dist.barrier()

        if rank != 0:
            return {"status": "worker_done"}

        if best_state_dict is not None:
            base_model.load_state_dict(best_state_dict)
        y_pred, y_true = predict_numpy(base_model, eval_loader, device)

        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        return {
            "status": "ok",
            "val_loss": float(best_val_loss),
            "mse": float(mse),
            "rmse": float(rmse),
            "mae": float(mae),
            "r2": float(r2),
            "y_pred": y_pred.tolist(),
            "y_true": y_true.tolist(),
            "state_dict": best_state_dict,
            "history": history,
            "params": {
                "input_dim": input_dim,
                "hidden_dims": list(hidden_dims),
                "dropout": dropout,
                "activation": activation,
                "lr": lr,
            },
        }
    finally:
        cleanup_distributed()



## 1.6 Optuna Objective Function

In [ ]:
def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 5)

    hidden_dims = [
        trial.suggest_int(f"hidden_{i}", 8, 512, log=True)
        for i in range(n_layers)
    ]

    dropout = trial.suggest_float("dropout", 0.0, 0.5)

    activation = trial.suggest_categorical(
        "activation",
        ["relu", "elu", "gelu", "leaky_relu"],
    )

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    result = run_with_torch_distributor(
        train_fn=train_ames_distributed,
        num_processes=2,
        local_mode=False,
        use_gpu=False,
        train_args=(
            X_train_np,
            y_train_np,
            X_test_np,
            y_test_np,
            input_dim,
            hidden_dims,
            dropout,
            activation,
            lr,
            32,
            200,
            20,
            1234,
        ),
    )

    return result["val_loss"]

## 2. Searching Hiper Parameters 

In [ ]:
import time

start = time.perf_counter()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

end = time.perf_counter()
elapsed = end - start

print(f"Tiempo total: {elapsed / 60:.2f} minutos")

In [ ]:
print("Mejor arquitectura encontrada:")
for k, v in study.best_trial.params.items():
    print(f"{k}: {v}")

study.best_trial.params

## 2.1 Final Traning with Best Hyper Parameters

In [ ]:
best = study.best_trial.params

best_hidden_dims = [
    best[f"hidden_{i}"]
    for i in range(best["n_layers"])
]

In [ ]:
final_result = run_with_torch_distributor(
    train_fn=train_ames_distributed,
    num_processes=2,
    local_mode=False,
    use_gpu=False,
    train_args=(
        X_train_np,
        y_train_np,
        X_test_np,
        y_test_np,
        input_dim,
        best_hidden_dims,
        best["dropout"],
        best["activation"],
        best["lr"],
        32,
        300,
        30,
        1234,
    ),
)

In [ ]:
final_model = MLPRegressor(
    input_dim=final_result["params"]["input_dim"],
    hidden_dims=final_result["params"]["hidden_dims"],
    dropout=final_result["params"]["dropout"],
    activation=final_result["params"]["activation"],
)

final_model.load_state_dict(final_result["state_dict"])
final_model.eval()


## 2.2 Final Evaluation

In [ ]:
y_pred = np.array(final_result["y_pred"])
y_true = np.array(final_result["y_true"])

mse = final_result["mse"]
rmse = final_result["rmse"]
mae = final_result["mae"]
r2 = final_result["r2"]

print(f"Test MSE : {mse:.2f}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test MAE : {mae:.2f}")
print(f"Test R²  : {r2:.4f}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_true, y_pred, alpha=0.5)
plt.plot(
    [y_true.min(), y_true.max()],
    [y_true.min(), y_true.max()],
    "r--",
)
plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.title("Actual vs Predicted Values")
plt.show()